# Colab verification gate - polyp segmentation data

This notebook is a **verification gate**, not a training notebook. It checks the Google Drive copy of the research project before any model training is allowed.

The notebook verifies:
- the benchmark directory structure;
- image and mask readability;
- image-mask dimensions and filename pairing;
- strict binary values in processed PNG masks;
- manifest row counts and target-file existence;
- train/validation disjointness;
- that validation is not accidentally one of the held-out test manifests.

A green gate means these checks passed in this Colab runtime. It does **not** prove clinical annotation quality, patient-level independence, or external dataset provenance.

In [1]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

drive_root = Path("/content/drive/MyDrive")
candidates = []

# Look only 1 or 2 levels deep to prevent slow recursive Drive operations
for first_level in drive_root.iterdir():
    if not first_level.is_dir():
        continue

    search_roots = [first_level]
    try:
        for second_level in first_level.iterdir():
            if second_level.is_dir():
                search_roots.append(second_level)
    except Exception:
        pass

    for search_root in search_roots:
        config_path = search_root / "configs" / "baseline_unet.yaml"
        script_path = search_root / "scripts" / "prepare_pranet_benchmark.py"

        if config_path.is_file() and script_path.is_file():
            candidates.append(search_root)

print("\nProject candidates:")
for candidate in candidates:
    print(candidate)

if len(candidates) == 0:
    # Fallback if it's directly under MyDrive
    config_path = drive_root / "configs" / "baseline_unet.yaml"
    if config_path.is_file():
        candidates.append(drive_root)
        print("Found project at MyDrive root:", drive_root)

if len(candidates) != 1:
    print("\n[WARNING] Could not find exactly one project folder. Setting a manual fallback.")
    # Check if a folder like 'weights' or 'Untitled folder' is indeed our root
    PROJECT_ROOT = Path('/content/drive/MyDrive')
else:
    PROJECT_ROOT = candidates[0]

print("\nUsing PROJECT_ROOT:", PROJECT_ROOT)

Mounted at /content/drive

Project candidates:

[WARNING] Could not find exactly one project folder. Setting a manual fallback.

Using PROJECT_ROOT: /content/drive/MyDrive


In [2]:
# Do not recursively walk all of Google Drive. Cell 2 already performs the bounded search.
PROJECT_ROOT = Path(PROJECT_ROOT)
required = [
    PROJECT_ROOT / 'configs/baseline_unet.yaml',
    PROJECT_ROOT / 'scripts/prepare_pranet_benchmark.py',
    PROJECT_ROOT / 'data/processed/manifests',
]
print('--- Project verification ---')
for path in required:
    print(('OK    ' if path.exists() else 'MISSING'), path)
    if not path.exists():
        raise FileNotFoundError(path)
print('PROJECT_ROOT_GATE=PASS')

--- Project verification ---
MISSING /content/drive/MyDrive/configs/baseline_unet.yaml


FileNotFoundError: /content/drive/MyDrive/configs/baseline_unet.yaml

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path(PROJECT_ROOT)
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Project not found: {PROJECT_ROOT}. Update PROJECT_ROOT to the Drive folder.'
    )

for required in ('scripts/prepare_pranet_benchmark.py', 'configs/baseline_unet.yaml', 'data/processed/manifests'):
    path = PROJECT_ROOT / required
    print(('OK   ' if path.exists() else 'MISS '), path)
    if not path.exists():
        raise FileNotFoundError(path)

## 1. Locate and inspect the raw benchmark

The expected raw layout is `data/raw/pranet_benchmark/TrainDataset` and `TestDataset`. The notebook does not silently choose a different directory.

In [ ]:
RAW_ROOT = PROJECT_ROOT / 'data/raw/pranet_benchmark'
required_dirs = [RAW_ROOT / 'TrainDataset', RAW_ROOT / 'TestDataset']
for path in required_dirs:
    print(('OK   ' if path.is_dir() else 'MISS '), path)
    if not path.is_dir():
        raise FileNotFoundError(path)

for path in sorted(RAW_ROOT.rglob('*')):
    if path.is_dir() and path.name in {'TrainDataset', 'TestDataset', 'image', 'images', 'masks'}:
        print(path.relative_to(PROJECT_ROOT))

## 2. Independently validate raw image-mask pairs

This check does not trust the preparation report. It opens every raw image and mask with Pillow, checks dimensions, and checks filename-stem pairing.

In [ ]:
from PIL import Image
from collections import Counter

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp'}

def files_in(path):
    return sorted(p for p in path.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def validate_pair_dirs(name, image_dir, mask_dir, expected):
    images = files_in(image_dir)
    masks = files_in(mask_dir)
    image_map = {p.stem: p for p in images}
    mask_map = {p.stem: p for p in masks}
    common = sorted(set(image_map) & set(mask_map))
    errors = []
    for stem in common:
        try:
            with Image.open(image_map[stem]) as image:
                image.load()
                image_size = image.size
            with Image.open(mask_map[stem]) as mask:
                mask.load()
                mask_size = mask.size
            if image_size != mask_size:
                errors.append((stem, image_size, mask_size))
        except Exception as exc:
            errors.append((stem, str(exc)))
    result = {
        'name': name, 'images': len(images), 'masks': len(masks),
        'paired': len(common), 'expected': expected,
        'unpaired_images': sorted(set(image_map) - set(mask_map)),
        'unpaired_masks': sorted(set(mask_map) - set(image_map)),
        'pair_errors': errors,
    }
    print(result)
    assert len(images) == expected and len(masks) == expected
    assert len(common) == expected
    assert not result['unpaired_images'] and not result['unpaired_masks']
    assert not errors
    return result

raw_results = []
raw_results.append(validate_pair_dirs(
    'TrainDataset', RAW_ROOT / 'TrainDataset/image', RAW_ROOT / 'TrainDataset/masks', 1450
))
for name, expected in [('CVC-300', 60), ('CVC-ClinicDB', 62), ('CVC-ColonDB', 380), ('ETIS-LaribPolypDB', 196), ('Kvasir', 100)]:
    raw_results.append(validate_pair_dirs(
        name, RAW_ROOT / f'TestDataset/{name}/images', RAW_ROOT / f'TestDataset/{name}/masks', expected
    ))
print('RAW_PAIR_GATE=PASS')

## 3. Independently validate processed masks

This explicitly measures whether binarization was needed. Do not claim that masks were repaired unless `non_binary_count` is greater than zero in this runtime.

In [ ]:
CLEAN_ROOT = PROJECT_ROOT / 'data/processed/cleaned_masks'
cleaned = sorted(CLEAN_ROOT.rglob('*.png'))
non_binary = []
unreadable = []
for path in cleaned:
    try:
        with Image.open(path) as image:
            values = set(image.convert('L').getdata())
        if not values.issubset({0, 255}):
            non_binary.append((str(path.relative_to(PROJECT_ROOT)), sorted(values)[:20]))
    except Exception as exc:
        unreadable.append((str(path), str(exc)))
    print('cleaned_png_count=', len(cleaned))
print('non_binary_count=', len(non_binary))
print('unreadable_count=', len(unreadable))
assert cleaned and not non_binary and not unreadable
print('PROCESSED_MASK_GATE=PASS')

## 4. Verify manifest counts, files, and disjointness

In [ ]:
import csv

MANIFEST_ROOT = PROJECT_ROOT / 'data/processed/manifests'
expected_counts = {
    'train_pranet.tsv': 1450,
    'test_cvc_colondb.tsv': 380,
    'test_etis_laribpolypdb.tsv': 196,
    'test_kvasir.tsv': 100,
    'test_cvc_clinicdb.tsv': 62,
    'test_cvc_300.tsv': 60,
    'train_pranet_train.tsv': 1232,
    'val_pranet.tsv': 218,
}
def read_manifest(name):
    path = MANIFEST_ROOT / name
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle, delimiter='\t'))
    assert rows and set(rows[0]) == {'image', 'mask'}
    return rows

manifests = {name: read_manifest(name) for name in expected_counts}
for name, rows in manifests.items():
    print(name, len(rows), 'expected', expected_counts[name])
    assert len(rows) == expected_counts[name]
    for row in rows:
        assert (PROJECT_ROOT / row['image']).is_file(), row['image']
        assert (PROJECT_ROOT / row['mask']).is_file(), row['mask']

train_images = {row['image'] for row in manifests['train_pranet_train.tsv']}
val_images = {row['image'] for row in manifests['val_pranet.tsv']}
train_masks = {row['mask'] for row in manifests['train_pranet_train.tsv']}
val_masks = {row['mask'] for row in manifests['val_pranet.tsv']}
assert train_images.isdisjoint(val_images)
assert train_masks.isdisjoint(val_masks)
assert 'test_kvasir.tsv' != 'val_pranet.tsv'
print('train_val_image_overlap=', len(train_images & val_images))
print('train_val_mask_overlap=', len(train_masks & val_masks))
print('MANIFEST_GATE=PASS')

## 5. Validate the baseline configuration

The baseline must use the generated train/validation manifests. External test manifests are for final evaluation only.

In [ ]:
import yaml
config = yaml.safe_load((PROJECT_ROOT / 'configs/baseline_unet.yaml').read_text(encoding='utf-8'))
data_config = config['data']
assert data_config['train_manifest'].endswith('train_pranet_train.tsv')
assert data_config['val_manifest'].endswith('val_pranet.tsv')
assert data_config['val_manifest'] not in data_config['test_manifests'].values()
print('train_manifest=', data_config['train_manifest'])
print('val_manifest=', data_config['val_manifest'])
print('CONFIG_GATE=PASS')

## Final decision

If every preceding cell passes, the data pipeline is approved for a **3-epoch baseline smoke test**. This notebook does not approve full training, paper claims, or the JPEG-artifact claim. Save the notebook output and the generated JSON report as the audit record.

In [ ]:
print('GREEN LIGHT: data and split gates passed for a 3-epoch smoke test only.')
print('Do not claim mask repair unless non_binary_count > 0 above.')